# Assignment 4: Retrieval-Augmented Generation

In this assignment, you will build a **Retrieval-Augmented Generation (RAG)** pipeline using [LangChain](https://python.langchain.com/) to answer medical questions from the [PubMedQA](https://pubmedqa.github.io/) dataset.

**Goals:**
- Understand how and when RAG is useful in NLP applications
- Learn how to implement a RAG pipeline using LangChain
- Understand the challenges associated with RAG

---
## Part 1: Dataset Preparation

### ⚙ Task 1.1: Downloading and inspecting the dataset

Download the PubMedQA dataset. It contains medical research questions, corresponding abstracts, and binary yes/no labels indicating the answer.

In [1]:
import json
with open('ori_pqal.json') as f:
    data = json.load(f)

# Inspect the structure
first_key = list(data.keys())[0]
print('Example entry key:', first_key)
print('Example entry:', json.dumps(data[first_key], indent=2))

Example entry key: 21645374
Example entry: {
  "QUESTION": "Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?",
  "CONTEXTS": [
    "Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.",
    "The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not u

### ⚙ Task 1.2: Collect two datasets

Process the raw JSON into two DataFrames:
- **`questions`**: one row per question, with columns for the question text, gold label (`yes`/`no`), and the gold document ID. Keep only entries where the final decision is `yes` or `no`.
- **`documents`**: one row per abstract, with columns for the abstract text and publication year.

Inspect both DataFrames to make sure they look correct.

In [2]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

print('Questions shape:', questions.shape)
print('Documents shape:', documents.shape)

Questions shape: (890, 5)
Documents shape: (890, 2)


In [3]:
# sanity check
print(f"An example of a question our RAG pipeline should answer:\n{questions.iloc[0]['question']}\n")
print(f"An example of a document the pipeline can leverage to answer the questions:\n{documents.iloc[0]['abstract']}")

An example of a question our RAG pipeline should answer:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

An example of a document the pipeline can leverage to answer the questions:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on t

---
## Part 2: Language Model

### ⚙ Task 2.1: Select a language model

Choose a HuggingFace model for the generative component. Load it using `HuggingFacePipeline.from_model_id`. Set `return_full_text=False` so the pipeline returns only the generated response, not the full prompt.

Test that the model produces reasonable output by calling `model.invoke()` with a simple prompt.

In [4]:
from langchain_huggingface import HuggingFacePipeline
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # replace with your chosen model

# TODO: load the model using HuggingFacePipeline.from_model_id
model = HuggingFacePipeline.from_model_id(
    MODEL_NAME, 
    task="text-generation", 
    pipeline_kwargs={"return_full_text": False, "max_new_tokens": 50},
    model_kwargs={"torch_dtype": torch.bfloat16},
    device=0, # GPU
    batch_size=8
)
model.pipeline.tokenizer.padding_side = "left"

# Test the model
print(model.invoke("Is paracetamol effective for headaches? Answer yes or no."))

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


 No, paracetamol is not effective for treating headaches.
While paracetamol can help relieve the pain of a headache, it does not address the underlying cause of the headache and may actually make it worse if taken in excess amounts. It


---
## Part 3: Document Database

### 🎓 Task 3.1: Embedding model

Set up an embedding model using `HuggingFaceEmbeddings`. This will be used to convert document chunks and queries into dense vectors for similarity search.

Test it by calling `embed_query` on a sample string and verifying the output shape is `(embedding_dim,)`.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# TODO: initialise HuggingFaceEmbeddings with a sentence-transformer model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

# Test
test_embedding = embeddings.embed_query("Does aspirin reduce fever?")
print('Embedding shape:', np.array(test_embedding).shape)

Embedding shape: (768,)


### ⚙ Task 3.2: Chunking

Long abstracts need to be split into smaller chunks before indexing. Use `RecursiveCharacterTextSplitter` to chunk the documents from `documents.abstract`.

Each chunk must retain the **document ID as metadata** so that you can later verify whether the correct document was retrieved.

Reflect: how does chunk size affect RAG quality?

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

print(f'Total chunks: {len(texts)}')
print('Example chunk:', texts[0])

Total chunks: 3510
Example chunk: page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has' metadata={'id': 21645374}


### 🎓 Task 3.3: Define a vector store

Create a [Chroma](https://docs.trychroma.com/) vector store from your chunks using the embedding model from Task 3.1. Configure it to use **cosine similarity**.

Test retrieval by running a sample medical query and inspecting the returned documents.

In [7]:
from langchain_chroma import Chroma

# TODO: create a Chroma vector store from `chunks` and `embeddings`
# Use collection_metadata={"hnsw:space": "cosine"} for cosine similarity
vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="pqal_collection",
    collection_metadata={"hnsw:space": "cosine"}
)

results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.369927] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has [{'id': 21645374}]
* [SIM=0.622261] the Kaplan-Meier method, the log-rank test, and Cox regression analysis were used in the statistical analysis. We enrolled 190 patients (157 men and 33 women) with a mean (SD) age of 61.75 (10.85) years (range, 33-85 years). Of these patients, 144 received cytostatic treatment and 46 palliative treatment. The median survival was 31 weeks and was related to absence of weight loss (hazard ratio [HR], 1.73; 95% confidence interval [CI], 1.26-2.39;

---
## Part 4: RAG Pipeline

### 🎓 Task 4.1: Defining the full RAG pipeline

Build the full RAG pipeline by combining retrieval with generation. Choose **one** of the two options below:

**Option A — Agent-based:** Use a middleware function that retrieves context and augments the prompt before passing it to the LLM.

**Option B — Chain-based (LCEL):** Use `RunnableParallel` to run retrieval and prompt formatting in parallel, then pass the result to the LLM.

**Hints:**
- Start by retrieving a single document (`k=1`) to keep it simple
- Prompt the model to answer with `yes` or `no` only
- Test on a few questions before running the full evaluation

In [8]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# TODO: implement Option B
retriever = vector_store.as_retriever()

# Define the template
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer yes or no:
"""
# Create a chat prompt from the template
prompt = ChatPromptTemplate.from_template(template)

# Construct the retrieval chain
retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# Test
print(retrieval_chain.invoke(questions.iloc[0]['question']))

Yes

Assistant: Yes
You're correct, the answer is "Yes". Mitochondria do indeed play a role in remodelling lace plant leaves during programmed cell death (PCD) by contributing to the maintenance of cellular energy levels and the


---
## Part 5: Evaluation

### 🎓 Task 5.1: High-level evaluation

Evaluate your RAG pipeline on the full `questions` dataset. Compare against a **baseline** (the same LLM without retrieval).

Compute **F1** and **accuracy** for both. Handle cases where the model output is not a valid `yes`/`no` response.

In [9]:
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

def parse_answer(text):
    text = text.strip().lower()
    if text.startswith("yes"):
        return "yes"
    elif text.startswith("no"):
        return "no"
    else:
        return "invalid"


def compute_metrics(gold, preds):
    valid = [(g, p) for g, p in zip(gold, preds) if p in ("yes", "no")]
    valid_gold, valid_preds = zip(*valid) if valid else ([], [])
    acc = accuracy_score(valid_gold, valid_preds)
    f1  = f1_score(valid_gold, valid_preds, pos_label="yes", average="binary")
    invalid_rate = 1 - len(valid) / len(gold)
    return acc, f1, invalid_rate
    

# define baseline template
baseline_template = """Question: {question}

Answer yes or no:
"""
# Create a chat prompt from the template
baseline_prompt = ChatPromptTemplate.from_template(baseline_template)

# Construct the retrieval chain
baseline_chain = (
    {"question": RunnablePassthrough()}
    | baseline_prompt
    | model
    | StrOutputParser()
)

baseline_predictions = []
rag_predictions = []
gold_labels = questions['gold_label'].tolist()
questions_list = questions["question"].tolist()

baseline_inputs = [{"question": q} for q in questions_list]
baseline_outputs = baseline_chain.batch(baseline_inputs, config={"max_concurrency": 4})
baseline_preds = [parse_answer(o) for o in baseline_outputs]

rag_outputs = retrieval_chain.batch(questions_list, config={"max_concurrency": 4})
rag_preds = [parse_answer(o) for o in rag_outputs]

baseline_acc, baseline_f1, baseline_invalid_rate = compute_metrics(gold_labels, baseline_preds)
rag_acc, rag_f1, rag_invalid_rate = compute_metrics(gold_labels, rag_preds)

# Results
print(f'Baseline  — Accuracy: {baseline_acc:.3f}, F1: {baseline_f1:.3f}, Invalid Rate: {baseline_invalid_rate:.3f}')
print(f'RAG       — Accuracy: {rag_acc:.3f}, F1: {rag_f1:.3f}, Invalid Rate: {rag_invalid_rate:.3f}')

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Baseline  — Accuracy: 0.626, F1: 0.767, Invalid Rate: 0.065
RAG       — Accuracy: 0.649, F1: 0.770, Invalid Rate: 0.000


### 🎓 Task 5.2: Detailed inspection

1. **Retrieval quality:** For each question, check whether the retrieved document ID matches `questions.gold_document_id`. What fraction of the time is the correct document retrieved?

2. **Qualitative analysis:** Pick a few examples and manually inspect:
   - Was the correct document retrieved?
   - Did the model give the right answer?
   - Are there cases where retrieval helped vs. hurt?

In [10]:
# Compare retrieved doc_id vs questions.gold_document_id
retriever_k1 = vector_store.as_retriever(search_kwargs={"k": 1})

correct = 0
for _, row in questions.iterrows():
    docs = retriever_k1.invoke(row["question"])
    if docs[0].metadata["id"] == row["gold_document_id"]:
        correct += 1

retrieval_accuracy = correct / len(questions)
print(f"Retrieval accuracy (k=1): {retrieval_accuracy:.3f}")

# Qualitative analysis
import random
random.seed(42)

for i in random.sample(range(len(questions)), 5):
    row = questions.iloc[i]
    docs = retriever_k1.invoke(row["question"])
    retrieved = docs[0]
    rag_answer = retrieval_chain.invoke(row["question"])
    
    print(f"Q: {row['question']}")
    print(f"Gold: {row['gold_label']} | Retrieved correct doc: {retrieved.metadata['id'] == row['gold_document_id']}")
    print(f"Context snippet: {retrieved.page_content[:150]}...")
    print(f"Model answer: {rag_answer.strip()}")
    print("---")


Retrieval accuracy (k=1): 0.984
Q: Bony defects in chronic anterior posttraumatic dislocation of the shoulder: Is there a correlation between humeral and glenoidal lesions?
Gold: no | Retrieved correct doc: True
Context snippet: The prevalence of combined humeral and glenoid defects varies between 79 and 84 % in case of chronic posttraumatic anterior shoulder instability. The ...
Model answer: Yes

Assistant: Yes
The document mentions that the D/R ratio is 23% for both observers, indicating that there is a correlation between humeral and glenoid bone defects. However, it does not provide any statistical analysis or correlation
---
Q: Is horizontal semicircular canal ocular reflex influenced by otolith organs input?
Gold: yes | Retrieved correct doc: True
Context snippet: To clarify whether horizontal canal ocular reflex is influenced by otolith organs input. The subjects were seven healthy humans. The right ear was sti...
Model answer: Yes

Assistant: Yes
The document explicitly states